Importando as bibliotecas e módulos necessários: 

In [15]:
import os
import pandas as pd
import numpy as np
import sys
import gc

Importando as bases necessárias: 

In [16]:
df_final = pd.read_parquet(r"C:\Users\emill\Downloads\TCC\processed\BASE_UNIF.parquet", engine="pyarrow")

In [17]:
df_final_2 = pd.read_parquet(r"C:\Users\emill\Downloads\TCC\processed\BASE_AED.parquet", engine="pyarrow")

Nessa etapa será feita a definição das variáveis a serem utilizadas:

• Dependente: nível de proficiência em matemática, categorizado;

• Independentes: a definir a partir da análise exploratória e de métricas de importância.
Serão explorados diferentes métodos para realizar essa seleção.

### 1. Categorização do nível de proficiência
Esta seção descreve o processo de categorização inicial a ser adotado: a classificação em 4 classes de proficiência (QEdu), e a divisão em 10 níveis (INEP), sendo eles:

• Insuficiente, para proficiências entre 0-224, Níveis 0 e 1

• Básico, para proficiências entre 225-299, Níveis 2, 3 e 4

• Proficiente, para proficiências entre 300-349, Níveis 5 e 6 

• Avançado, para proficiências a partir de 350, Níveis 7, 8 e 9

In [18]:
bins = [-float('inf'), 200, 225, 250, 275, 300, 325, 350, 375, 400, float('inf')]
labels_numericos = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
labels_proficiencia = [
    'Insuficiente', 
    'Básico',       
    'Proficiente',   
    'Avançado'     
]

### 2. Seleção de características baseada na Análise Exploratória

#### 2019
Colunas selecionadas: LOCALIZACAO, COR/RACA, ESCOL_MAE, FREQUENCIA_CONVERSA, QTD_COMPUTADOR, QTD_CARRO, GARAGEM, IDADE_INTROD_ESC, REPROVACAO, TEMPO_ESTUDO, TEMPO_TRAB_DOMES, POS_EF

In [19]:
df_2019 = df_final[df_final["ANO"] == 2019]

df_2019['NIVEL_PROFICIENCIA'] = pd.cut(df_2019['PROFICIENCIA_SAEB'], bins=bins, labels=labels_numericos, right=False)

nivel_to_proficiencia = {0: 'Insuficiente', 1: 'Insuficiente', 2: 'Básico', 3: 'Básico', 4: 'Básico',
                         5: 'Proficiente', 6: 'Proficiente', 7: 'Avançado', 8: 'Avançado', 9: 'Avançado'}

df_2019['PROFICIENCIA_DESCRICAO'] = df_2019['NIVEL_PROFICIENCIA'].map(nivel_to_proficiencia)

colunas = [
    'LOCALIZACAO', 'COR_RACA', 'ESCOL_MAE', 'FREQUENCIA_CONVERSA', 'QTD_COMPUTADOR', 'QTD_CARRO', 'GARAGEM', 'IDADE_INTROD_ESC',     
    'REPROVACAO', 'TEMPO_ESTUDO', 'TEMPO_TRAB_DOMES', 'POS_EF', 'PROFICIENCIA_DESCRICAO'

]

df_2019 = df_2019[colunas]
df_2019.head()

,LOCALIZACAO,COR_RACA,ESCOL_MAE,FREQUENCIA_CONVERSA,QTD_COMPUTADOR,QTD_CARRO,GARAGEM,IDADE_INTROD_ESC,REPROVACAO,TEMPO_ESTUDO,TEMPO_TRAB_DOMES,POS_EF,PROFICIENCIA_DESCRICAO
0,1,Branca,Fundamental incompleto,Às vezes,Não respondeu,1 ou 2,Sim,Entre 4 e 7,Sim,Menos de 1 hora,Mais de 2 horas,Estudar e trabalhar,Básico
1,1,Parda,Superior completo,Sempre,0,1 ou 2,Sim,Entre 4 e 7,Não,Menos de 1 hora,Mais de 2 horas,Estudar e trabalhar,Básico
2,1,Parda,Não sabe,Às vezes,0,0,Não,Entre 4 e 7,Não,Entre 1 e 2 horas,Mais de 2 horas,Não sabe,Básico
3,1,Parda,Superior completo,Às vezes,1 ou 2,1 ou 2,Sim,Entre 4 e 7,Não,Entre 1 e 2 horas,Mais de 2 horas,Não sabe,Básico
4,1,Branca,Fundamental incompleto,Nunca,0,1 ou 2,Sim,Entre 4 e 7,Não,Menos de 1 hora,Não respondeu,Estudar e trabalhar,Básico


In [20]:
# Verificando a distribuição das classes 
class_distribution = df_2019['PROFICIENCIA_DESCRICAO'].value_counts()

# Total de registros no dataframe
total_records = len(df_2019)

print("Distribuição de registros:")
for classe, quantidade in class_distribution.items():
    porcentagem = (quantidade / total_records) * 100
    print(f"Classe {classe}: {quantidade} registros ({porcentagem:.2f}%)")

# Verificando se a soma total bate
print(f"\nSoma total de registros: {class_distribution.sum()} (Esperado: {total_records})")


Distribuição de registros:
Classe Básico: 1042352 registros (54.52%)
Classe Insuficiente: 507526 registros (26.55%)
Classe Proficiente: 315123 registros (16.48%)
Classe Avançado: 46930 registros (2.45%)

Soma total de registros: 1911931 (Esperado: 1911931)


#### 2021
Colunas selecionadas: LOCALIZACAO, SEXO, COR/RACA, ESCOL_MAE, FREQUENCIA_CONVERSA, QTD_COMPUTADOR, QTD_CARRO, GARAGEM, IDADE_INTROD_ESC, REPROVACAO, TEMPO_ESTUDO, TEMPO_TRAB_DOMES, POS_EF

In [21]:
df_2021 = df_final_2[df_final_2["ANO"] == 2021]

df_2021['NIVEL_PROFICIENCIA'] = pd.cut(df_2021['PROFICIENCIA_SAEB'], bins=bins, labels=labels_numericos, right=False)

nivel_to_proficiencia = {0: 'Insuficiente', 1: 'Insuficiente', 2: 'Básico', 3: 'Básico', 4: 'Básico',
                         5: 'Proficiente', 6: 'Proficiente', 7: 'Avançado', 8: 'Avançado', 9: 'Avançado'}

df_2021['PROFICIENCIA_DESCRICAO'] = df_2021['NIVEL_PROFICIENCIA'].map(nivel_to_proficiencia)

colunas = [
    'SEXO', 'QTD_COMPUTADOR', 'ESCOL_MAE', 'POS_EF', 'QTD_CARRO', 'FREQUENCIA_CONVERSA', 'COR/RACA', 'IDADE_INTROD_ESC', 'REPROVACAO', 'PROFICIENCIA_DESCRICAO'    
]
# 'LOCALIZACAO', 'SEXO', 'COR/RACA', 'ESCOL_MAE', 'FREQUENCIA_CONVERSA', 'QTD_COMPUTADOR', 'QTD_CARRO', 'GARAGEM', 'IDADE_INTROD_ESC', 'REPROVACAO', 'TEMPO_ESTUDO', 'TEMPO_TRAB_DOMES', 'POS_EF', 'PROFICIENCIA_DESCRICAO' 
df_2021 = df_2021[colunas]
df_2021.head()

,SEXO,QTD_COMPUTADOR,ESCOL_MAE,POS_EF,QTD_CARRO,FREQUENCIA_CONVERSA,COR/RACA,IDADE_INTROD_ESC,REPROVACAO,PROFICIENCIA_DESCRICAO
0,Masculino,0,Médio completo,Estudar e trabalhar,0,Às vezes,Branca,3 ou menos,Não,Básico
1,Masculino,1 ou 2,Não sabe,Somente trabalhar,0,Sempre,Parda,Entre 4 e 7,Sim,Insuficiente
2,Feminino,1 ou 2,Fundamental incompleto,Estudar e trabalhar,Não respondeu,Às vezes,Branca,Entre 4 e 7,Não respondeu,Insuficiente
3,Feminino,0,Não sabe,Estudar e trabalhar,1 ou 2,Nunca,Parda,Entre 4 e 7,Sim,Insuficiente
4,Feminino,0,Não sabe,Estudar e trabalhar,0,Às vezes,Outra,Entre 4 e 7,Não,Básico


In [22]:
# Verificando a distribuição das classes 
class_distribution = df_2021['PROFICIENCIA_DESCRICAO'].value_counts()

# Total de registros no dataframe
total_records = len(df_2021)

print("Distribuição de registros:")
for classe, quantidade in class_distribution.items():
    porcentagem = (quantidade / total_records) * 100
    print(f"Classe {classe}: {quantidade} registros ({porcentagem:.2f}%)")

# Verificando se a soma total bate
print(f"\nSoma total de registros: {class_distribution.sum()} (Esperado: {total_records})")


Distribuição de registros:
Classe Básico: 1016280 registros (53.88%)
Classe Insuficiente: 569072 registros (30.17%)
Classe Proficiente: 270061 registros (14.32%)
Classe Avançado: 30766 registros (1.63%)

Soma total de registros: 1886179 (Esperado: 1886179)


#### 2023
Colunas selecionadas: LOCALIZACAO, SEXO, COR/RACA, ESCOL_MAE, FREQUENCIA_CONVERSA, QTD_COMPUTADOR, QTD_CARRO, GARAGEM, IDADE_INTROD_ESC, REPROVACAO, TEMPO_ESTUDO, TEMPO_TRAB_DOMES, POS_EF

In [23]:
df_2023 = df_final_2[df_final_2["ANO"] == 2023]

df_2023['NIVEL_PROFICIENCIA'] = pd.cut(df_2023['PROFICIENCIA_SAEB'], bins=bins, labels=labels_numericos, right=False)

nivel_to_proficiencia = {0: 'Insuficiente', 1: 'Insuficiente', 2: 'Básico', 3: 'Básico', 4: 'Básico',
                         5: 'Proficiente', 6: 'Proficiente', 7: 'Avançado', 8: 'Avançado', 9: 'Avançado'}

df_2023['PROFICIENCIA_DESCRICAO'] = df_2023['NIVEL_PROFICIENCIA'].map(nivel_to_proficiencia)

colunas = [
    'LOCALIZACAO', 'SEXO', 'COR/RACA', 'ESCOL_MAE', 'FREQUENCIA_CONVERSA', 'QTD_COMPUTADOR', 'QTD_CARRO', 'GARAGEM', 'IDADE_INTROD_ESC',     
    'REPROVACAO', 'TEMPO_ESTUDO', 'TEMPO_TRAB_DOMES', 'POS_EF', 'PROFICIENCIA_DESCRICAO'

]

df_2023 = df_2023[colunas]
df_2023.head()

,LOCALIZACAO,SEXO,COR/RACA,ESCOL_MAE,FREQUENCIA_CONVERSA,QTD_COMPUTADOR,QTD_CARRO,GARAGEM,IDADE_INTROD_ESC,REPROVACAO,TEMPO_ESTUDO,TEMPO_TRAB_DOMES,POS_EF,PROFICIENCIA_DESCRICAO
1886179,1,Masculino,Parda,Médio completo,Às vezes,1 ou 2,1 ou 2,Sim,Entre 4 e 7,Não,Não usa o tempo para isso,Mais de 2 horas,Somente trabalhar,Insuficiente
1886180,1,Feminino,Preta,Não sabe,Não respondeu,Não respondeu,Não respondeu,Não respondeu,Entre 4 e 7,Não,Não respondeu,Não respondeu,Somente estudar,Insuficiente
1886181,1,Feminino,Parda,Fundamental incompleto,Não respondeu,0,1 ou 2,Não,Entre 4 e 7,Sim,Menos de 1 hora,Mais de 2 horas,Somente estudar,Básico
1886182,1,Masculino,Não respondeu,Fundamental incompleto,Nunca,1 ou 2,1 ou 2,Não,Entre 4 e 7,Sim,Menos de 1 hora,Entre 1 e 2 horas,Estudar e trabalhar,Básico
1886183,1,Feminino,Parda,Médio completo,Sempre,1 ou 2,1 ou 2,Sim,Entre 4 e 7,Não,Menos de 1 hora,Mais de 2 horas,Estudar e trabalhar,Básico


In [24]:
# Verificando a distribuição das classes 
class_distribution = df_2023['PROFICIENCIA_DESCRICAO'].value_counts()

# Total de registros no dataframe
total_records = len(df_2023)

print("Distribuição de registros:")
for classe, quantidade in class_distribution.items():
    porcentagem = (quantidade / total_records) * 100
    print(f"Classe {classe}: {quantidade} registros ({porcentagem:.2f}%)")

# Verificando se a soma total bate
print(f"\nSoma total de registros: {class_distribution.sum()} (Esperado: {total_records})")


Distribuição de registros:
Classe Básico: 1052094 registros (51.26%)
Classe Insuficiente: 655152 registros (31.92%)
Classe Proficiente: 294108 registros (14.33%)
Classe Avançado: 51302 registros (2.50%)

Soma total de registros: 2052656 (Esperado: 2052656)


--------------------------- Modelagem Preditiva ----------------------------

In [8]:
#!pip install seaborn
#!pip install scipy
#!pip install statsmodels scikit-learn
#!pip install imbalanced-learn

In [9]:
import scipy.stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
#from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler

#### 2019

In [25]:
# =====================================
# Ajustando a variável alvo
# =====================================

df_2019['PROFICIENCIA_DESCRICAO'] = (
    df_2019['PROFICIENCIA_DESCRICAO']
    .replace({
        'Básico': 'Insuficiente',
        'Insuficiente': 'Insuficiente',
        'Proficiente': 'Proficiente',
        'Avançado': 'Proficiente'
    })
)

# =====================================
# Variável dependente
# =====================================

y = df_2019['PROFICIENCIA_DESCRICAO'].map({
    'Insuficiente': 0,
    'Proficiente': 1
})

# =====================================
# Variáveis independentes
# =====================================

X = df_2019.drop(columns=['PROFICIENCIA_DESCRICAO'])

# Dummies
X = pd.get_dummies(X, drop_first=True)

# Remove colunas constantes
X = X.loc[:, X.nunique() > 1]

# Remove NA
X = X.dropna()

# Alinha y
y = y.loc[X.index]

# Conversão
X = X.astype(float)
y = y.astype(int)

# Intercepto
X = sm.add_constant(X)

# =====================================
# Divisão treino/teste
# =====================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# =====================================
# Distribuição das classes
# =====================================

print("Distribuição das classes:")
print(y_train.value_counts())

# =====================================
# Cálculo dos pesos
# =====================================

n_total = len(y_train)

n0 = (y_train == 0).sum()
n1 = (y_train == 1).sum()

peso_0 = n_total / (2 * n0)
peso_1 = n_total / (2 * n1)

print(f"\nPeso classe 0: {peso_0:.4f}")
print(f"Peso classe 1: {peso_1:.4f}")

weights = y_train.map({
    0: peso_0,
    1: peso_1
})

# =====================================
# Modelo Logístico Ponderado
# =====================================

modelo = sm.GLM(
    y_train,
    X_train,
    family=sm.families.Binomial(),
    freq_weights=weights
).fit()

# =====================================
# Resumo
# =====================================

print(modelo.summary())

Distribuição das classes:
PROFICIENCIA_DESCRICAO
0    1239902
1     289642
Name: count, dtype: int64

Peso classe 0: 0.6168
Peso classe 1: 2.6404
                   Generalized Linear Model Regression Results                    
Dep. Variable:     PROFICIENCIA_DESCRICAO   No. Observations:              1529544
Model:                                GLM   Df Residuals:               1529504.00
Model Family:                    Binomial   Df Model:                           39
Link Function:                      Logit   Scale:                          1.0000
Method:                              IRLS   Log-Likelihood:            -9.3995e+05
Date:                    Wed, 17 Jun 2026   Deviance:                   1.8799e+06
Time:                            16:16:32   Pearson chi2:                 1.53e+06
No. Iterations:                         5   Pseudo R-squ. (CS):             0.1455
Covariance Type:                nonrobust                                         
                        

In [26]:
coef = modelo.params

odds_ratios = np.exp(coef)

p_values = modelo.pvalues

resultado = pd.DataFrame({
    'Feature': coef.index,
    'Coefficient': coef.values,
    'Odds Ratio': odds_ratios.values,
    'p-value': p_values.values
})

resultado = resultado.sort_values(
    by='Odds Ratio',
    ascending=False
)

print(resultado)

                                       Feature  Coefficient  Odds Ratio  \
0                                        const     0.643738    1.903583   
16                    QTD_COMPUTADOR_3 ou mais     0.413357    1.511885   
25              IDADE_INTROD_ESC_Não respondeu     0.336500    1.400039   
15                       QTD_COMPUTADOR_1 ou 2     0.315376    1.370775   
11                 ESCOL_MAE_Superior completo     0.291027    1.337801   
38                      POS_EF_Somente estudar     0.272464    1.313196   
20                     QTD_CARRO_Não respondeu     0.221939    1.248496   
18                            QTD_CARRO_1 ou 2     0.208868    1.232282   
8                     ESCOL_MAE_Médio completo     0.159421    1.172832   
22                                 GARAGEM_Sim     0.144204    1.155120   
3                       COR_RACA_Não respondeu     0.131538    1.140581   
19                         QTD_CARRO_3 ou mais     0.092539    1.096956   
26                    REP

#### 2021

In [27]:
# =====================================
# Ajustando a variável alvo
# =====================================

df_2021['PROFICIENCIA_DESCRICAO'] = (
    df_2021['PROFICIENCIA_DESCRICAO']
    .replace({
        'Básico': 'Insuficiente',
        'Insuficiente': 'Insuficiente',
        'Proficiente': 'Proficiente',
        'Avançado': 'Proficiente'
    })
)

# =====================================
# Variável dependente
# =====================================

y = df_2021['PROFICIENCIA_DESCRICAO'].map({
    'Insuficiente': 0,
    'Proficiente': 1
})

# =====================================
# Variáveis independentes
# =====================================

X = df_2021.drop(columns=['PROFICIENCIA_DESCRICAO'])

# Dummies
X = pd.get_dummies(X, drop_first=True)

# Remove colunas constantes
X = X.loc[:, X.nunique() > 1]

# Remove NA
X = X.dropna()

# Alinha y
y = y.loc[X.index]

# Conversão
X = X.astype(float)
y = y.astype(int)

# Intercepto
X = sm.add_constant(X)

# =====================================
# Divisão treino/teste
# =====================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# =====================================
# Distribuição das classes
# =====================================

print("Distribuição das classes:")
print(y_train.value_counts())

# =====================================
# Cálculo dos pesos
# =====================================

n_total = len(y_train)

n0 = (y_train == 0).sum()
n1 = (y_train == 1).sum()

peso_0 = n_total / (2 * n0)
peso_1 = n_total / (2 * n1)

print(f"\nPeso classe 0: {peso_0:.4f}")
print(f"Peso classe 1: {peso_1:.4f}")

weights = y_train.map({
    0: peso_0,
    1: peso_1
})

# =====================================
# Modelo Logístico Ponderado
# =====================================

modelo = sm.GLM(
    y_train,
    X_train,
    family=sm.families.Binomial(),
    freq_weights=weights
).fit()

# =====================================
# Resumo
# =====================================

print(modelo.summary())

Distribuição das classes:
PROFICIENCIA_DESCRICAO
0    1268281
1     240662
Name: count, dtype: int64

Peso classe 0: 0.5949
Peso classe 1: 3.1350
                   Generalized Linear Model Regression Results                    
Dep. Variable:     PROFICIENCIA_DESCRICAO   No. Observations:              1508943
Model:                                GLM   Df Residuals:                  1508912
Model Family:                    Binomial   Df Model:                           30
Link Function:                      Logit   Scale:                          1.0000
Method:                              IRLS   Log-Likelihood:            -9.1274e+05
Date:                    Wed, 17 Jun 2026   Deviance:                   1.8255e+06
Time:                            16:33:33   Pearson chi2:                 1.51e+06
No. Iterations:                         5   Pseudo R-squ. (CS):             0.1618
Covariance Type:                nonrobust                                         
                        

In [28]:
coef = modelo.params

odds_ratios = np.exp(coef)

p_values = modelo.pvalues

resultado = pd.DataFrame({
    'Feature': coef.index,
    'Coefficient': coef.values,
    'Odds Ratio': odds_ratios.values,
    'p-value': p_values.values
})

resultado = resultado.sort_values(
    by='Odds Ratio',
    ascending=False
)

print(resultado)

                              Feature  Coefficient  Odds Ratio        p-value
1                      SEXO_Masculino     0.617072    1.853493   0.000000e+00
4            QTD_COMPUTADOR_3 ou mais     0.506324    1.659181   0.000000e+00
3               QTD_COMPUTADOR_1 ou 2     0.397989    1.488827   0.000000e+00
10        ESCOL_MAE_Superior completo     0.369314    1.446742   0.000000e+00
2                  SEXO_Não respondeu     0.292145    1.339297   8.661370e-49
15                   QTD_CARRO_1 ou 2     0.248163    1.281669   0.000000e+00
7            ESCOL_MAE_Médio completo     0.219045    1.244887  5.124547e-290
13             POS_EF_Somente estudar     0.173985    1.190038  1.799646e-250
28     IDADE_INTROD_ESC_Não respondeu     0.087427    1.091362   2.617602e-09
20       FREQUENCIA_CONVERSA_Às vezes     0.061365    1.063287   2.641165e-27
16                QTD_CARRO_3 ou mais     0.036342    1.037010   3.761130e-04
27       IDADE_INTROD_ESC_Entre 4 e 7     0.017940    1.018102  

#### 2023

In [29]:
# =====================================
# Ajustando a variável alvo
# =====================================

df_2023['PROFICIENCIA_DESCRICAO'] = (
    df_2023['PROFICIENCIA_DESCRICAO']
    .replace({
        'Básico': 'Insuficiente',
        'Insuficiente': 'Insuficiente',
        'Proficiente': 'Proficiente',
        'Avançado': 'Proficiente'
    })
)

# =====================================
# Variável dependente
# =====================================

y = df_2023['PROFICIENCIA_DESCRICAO'].map({
    'Insuficiente': 0,
    'Proficiente': 1
})

# =====================================
# Variáveis independentes
# =====================================

X = df_2023.drop(columns=['PROFICIENCIA_DESCRICAO'])

# Dummies
X = pd.get_dummies(X, drop_first=True)

# Remove colunas constantes
X = X.loc[:, X.nunique() > 1]

# Remove NA
X = X.dropna()

# Alinha y
y = y.loc[X.index]

# Conversão
X = X.astype(float)
y = y.astype(int)

# Intercepto
X = sm.add_constant(X)

# =====================================
# Divisão treino/teste
# =====================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# =====================================
# Distribuição das classes
# =====================================

print("Distribuição das classes:")
print(y_train.value_counts())

# =====================================
# Cálculo dos pesos
# =====================================

n_total = len(y_train)

n0 = (y_train == 0).sum()
n1 = (y_train == 1).sum()

peso_0 = n_total / (2 * n0)
peso_1 = n_total / (2 * n1)

print(f"\nPeso classe 0: {peso_0:.4f}")
print(f"Peso classe 1: {peso_1:.4f}")

weights = y_train.map({
    0: peso_0,
    1: peso_1
})

# =====================================
# Modelo Logístico Ponderado
# =====================================

modelo = sm.GLM(
    y_train,
    X_train,
    family=sm.families.Binomial(),
    freq_weights=weights
).fit()

# =====================================
# Resumo
# =====================================

print(modelo.summary())

Distribuição das classes:
PROFICIENCIA_DESCRICAO
0    1365796
1     276328
Name: count, dtype: int64

Peso classe 0: 0.6012
Peso classe 1: 2.9713
                   Generalized Linear Model Regression Results                    
Dep. Variable:     PROFICIENCIA_DESCRICAO   No. Observations:              1642124
Model:                                GLM   Df Residuals:               1642081.00
Model Family:                    Binomial   Df Model:                           42
Link Function:                      Logit   Scale:                          1.0000
Method:                              IRLS   Log-Likelihood:            -1.0023e+06
Date:                    Wed, 17 Jun 2026   Deviance:                   2.0046e+06
Time:                            16:36:51   Pearson chi2:                 1.65e+06
No. Iterations:                         5   Pseudo R-squ. (CS):             0.1526
Covariance Type:                nonrobust                                         
                        

In [30]:
coef = modelo.params

odds_ratios = np.exp(coef)

p_values = modelo.pvalues

resultado = pd.DataFrame({
    'Feature': coef.index,
    'Coefficient': coef.values,
    'Odds Ratio': odds_ratios.values,
    'p-value': p_values.values
})

resultado = resultado.sort_values(
    by='Odds Ratio',
    ascending=False
)

print(resultado)

                                       Feature  Coefficient  Odds Ratio  \
4                           SEXO_Não respondeu     0.699688    2.013125   
2                               SEXO_Masculino     0.645514    1.906966   
19                    QTD_COMPUTADOR_3 ou mais     0.477887    1.612664   
14                 ESCOL_MAE_Superior completo     0.343480    1.409846   
18                       QTD_COMPUTADOR_1 ou 2     0.325953    1.385350   
3                       SEXO_Não quis declarar     0.305850    1.357779   
0                                        const     0.268484    1.307980   
24                       GARAGEM_Não respondeu     0.234349    1.264085   
41                      POS_EF_Somente estudar     0.226209    1.253838   
11                    ESCOL_MAE_Médio completo     0.221970    1.248534   
21                            QTD_CARRO_1 ou 2     0.177580    1.194323   
28              IDADE_INTROD_ESC_Não respondeu     0.165450    1.179924   
25                       